In [5]:
# -*- coding: utf-8 -*-
"""
Optimized 2D PINN - Fair Comparison Version
Aligned with 1D/2D FL-DAE strict LHS-based testing and timing protocols.
- Removed type hints to prevent NameError.
- Includes exact NUM_SAMPLES evaluations.
- T_eval timing separated from e2/einf computations.
- Supports multiple mu loop iterations gracefully.
"""

import time
import os
import math
import random
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.autograd import grad
from scipy.stats import qmc
from scipy.spatial import cKDTree

# =============================================================================
# Basic settings
# =============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = False

SEEDS = [33, 99, 202, 1234, 5678, 9999]
MU_LIST = [0.01, 0.001, 0.0001]

X_MIN, X_MAX = -2.0, 2.0
Y_MIN, Y_MAX = -4.0, 4.0
T_FINAL = 1.0

# 采样点数
N_F = 4000
N_B = 2000
N_I = 4000
EPOCHS = 30000

# LHS 测试设置
NUM_SAMPLES = 10000
LHS_SEED = 1234
EVAL_WARMUP = 20
EVAL_REPEAT = 200

BASE_PATH = "."
SAVE_LHS_PREDICTION = True
METHOD_NAME = "PINN"

# =============================================================================
# Utilities
# =============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def mean_std(values):
    arr = np.asarray(values, dtype=float)
    if len(arr) <= 1: return np.nanmean(arr), 0.0
    return np.nanmean(arr), np.nanstd(arr, ddof=1)

# =============================================================================
# 网络结构与物理函数
# =============================================================================
class PINN(nn.Module):
    def __init__(self, layers):
        super(PINN, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            layer = nn.Linear(layers[i], layers[i + 1])
            nn.init.xavier_normal_(layer.weight)  
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)
            self.layers.append(layer)
        self.activation = nn.Tanh()

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = self.activation(layer(x))
        return self.layers[-1](x)

def f_star(x, y):
    return torch.cos(np.pi * x / 4) * torch.cos(np.pi * y / 4)

def u_init(x, y, mu):
    return 3 * torch.tanh(x / mu + y) - 1

def compute_loss(model, X_f, X_b_left, X_b_right, X_b_bottom, X_b_top, X_i, U_i_target, mu):
    losses = {}
    
    u_internal = model(X_f)
    grads = grad(u_internal.sum(), X_f, create_graph=True)[0]
    u_x = grads[:, 0:1]
    u_y = grads[:, 1:2]
    u_t = grads[:, 2:3]
    
    u_xx = grad(u_x.sum(), X_f, create_graph=True)[0][:, 0:1]
    u_yy = grad(u_y.sum(), X_f, create_graph=True)[0][:, 1:2]

    f = f_star(X_f[:, 0:1], X_f[:, 1:2])
    pde_residual = mu * (u_xx + u_yy) - u_t + u_internal * (u_x + u_y) - f
    losses['pde'] = (pde_residual ** 2).mean()

    # 边界条件 (y方向软周期)
    u_left = model(X_b_left)
    u_right = model(X_b_right)
    u_bottom = model(X_b_bottom) 
    u_top = model(X_b_top)       
    
    losses['bc'] = ((u_left - (-4)) ** 2 + (u_right - 2) ** 2 + (u_bottom - u_top) ** 2).mean()

    u_initial = model(X_i)
    losses['ic'] = ((u_initial.squeeze() - U_i_target) ** 2).mean()

    return sum(losses.values()), losses

# =============================================================================
# LHS Reference Data and Evaluation Loaders
# =============================================================================
def get_target_col(df):
    if "u" in df.columns: return "u"
    if "u0" in df.columns: return "u0"
    return df.columns[-1]

def load_true_solution(mu):
    mu_id = round(-math.log10(mu))
    filename = f"2d_U0_all_t_u_x_y_t_mu{mu_id}_101_101_101_Mathematica_620.csv"
    path = os.path.join(BASE_PATH, filename)
    if not os.path.exists(path): raise FileNotFoundError(f"Missing reference file: {filename}")
    df = pd.read_csv(path)
    df.columns = [str(col).lower().strip() for col in df.columns]
    df = df.sort_values(by=["t", "x", "y"]).reset_index(drop=True)
    return df, filename

def build_or_load_lhs_test_set(mu):
    df_true, _ = load_true_solution(mu)
    index_file = f"2d_LHS_sample_indices_mu{mu:.0e}.npy"

    sample_indices = None
    if os.path.exists(index_file):
        loaded = np.load(index_file)
        valid = (len(loaded) == NUM_SAMPLES and len(np.unique(loaded)) == NUM_SAMPLES and np.max(loaded) < len(df_true))
        if valid:
            sample_indices = loaded
            print(f"[mu={mu}] Loaded valid LHS indices from {index_file}.")

    if sample_indices is None:
        t_min, t_max = df_true["t"].min(), df_true["t"].max()
        x_min, x_max = df_true["x"].min(), df_true["x"].max()
        y_min, y_max = df_true["y"].min(), df_true["y"].max()
        all_points = df_true[["t", "x", "y"]].values
        kdtree = cKDTree(all_points)
        selected, used, batch_id = [], set(), 0
        
        while len(selected) < NUM_SAMPLES and batch_id < 100:
            sampler = qmc.LatinHypercube(d=3, seed=LHS_SEED + batch_id)
            lhs_scaled = qmc.scale(sampler.random(n=NUM_SAMPLES), [t_min, x_min, y_min], [t_max, x_max, y_max])
            _, candidate_indices = kdtree.query(lhs_scaled)
            for idx in candidate_indices:
                idx = int(idx)
                if idx not in used:
                    used.add(idx)
                    selected.append(idx)
                    if len(selected) == NUM_SAMPLES: break
            batch_id += 1

        if len(selected) < NUM_SAMPLES:
            remaining = np.setdiff1d(np.arange(len(df_true)), np.asarray(selected, dtype=int), assume_unique=False)
            fill = np.random.default_rng(LHS_SEED).choice(remaining, size=NUM_SAMPLES - len(selected), replace=False)
            selected.extend([int(i) for i in fill])

        sample_indices = np.asarray(selected, dtype=int)
        np.save(index_file, sample_indices)

    t_lhs_np = df_true.iloc[sample_indices]["t"].values.reshape(-1, 1)
    x_lhs_np = df_true.iloc[sample_indices]["x"].values.reshape(-1, 1)
    y_lhs_np = df_true.iloc[sample_indices]["y"].values.reshape(-1, 1)
    true_col = get_target_col(df_true)
    true_lhs_np = df_true.iloc[sample_indices][true_col].values.reshape(-1)

    return {
        "net_input": torch.tensor(np.hstack([x_lhs_np, y_lhs_np, t_lhs_np]), dtype=torch.float32, device=device),
        "true_lhs_np": true_lhs_np,
        "t_np": t_lhs_np, "x_np": x_lhs_np, "y_np": y_lhs_np, "n_test": len(sample_indices),
    }

def compute_error(true_u, pred_u):
    diff = pred_u - true_u
    e2 = np.linalg.norm(diff) / np.linalg.norm(true_u)
    einf = np.max(np.abs(diff))
    return e2, einf

# =============================================================================
# Main Program
# =============================================================================
if __name__ == "__main__":
    print("\n" + "="*80)
    print(f"Starting 2D {METHOD_NAME} Benchmark with Strict LHS-based Timing")
    print(f"Device: {device} | LHS test points: {NUM_SAMPLES}")
    print("="*80 + "\n")

    lhs_data = {mu: build_or_load_lhs_test_set(mu) for mu in MU_LIST}
    metrics = {mu: [] for mu in MU_LIST}

    for mu in MU_LIST:
        print("\n" + "=" * 80)
        print(f"Starting 2D {METHOD_NAME} for mu={mu}")
        print("=" * 80)
        
        data_mu = lhs_data[mu]
        net_input_eval = data_mu["net_input"]
        true_np = data_mu["true_lhs_np"]
        n_test = data_mu["n_test"]

        for seed in SEEDS:
            print(f"\n--- Running Seed: {seed} ---")
            set_seed(seed)
            
            # 1. 内部配置点
            x_f_raw = torch.rand(N_F, 1, device=device) * 4 - 2
            y_f_raw = torch.rand(N_F, 1, device=device) * 8 - 4
            t_f_raw = torch.rand(N_F, 1, device=device)
            X_f = torch.cat([x_f_raw, y_f_raw, t_f_raw], dim=1).requires_grad_(True)
            
            # 2. 边界点
            x_b_raw = torch.rand(N_B, 1, device=device) * 4 - 2
            y_b_raw = torch.rand(N_B, 1, device=device) * 8 - 4 
            t_b_raw = torch.rand(N_B, 1, device=device)
            
            X_b_left = torch.cat([torch.full_like(y_b_raw, -2.0), y_b_raw, t_b_raw], dim=1)
            X_b_right = torch.cat([torch.full_like(y_b_raw, 2.0), y_b_raw, t_b_raw], dim=1)
            X_b_bottom = torch.cat([x_b_raw, torch.full_like(x_b_raw, -4.0), t_b_raw], dim=1) 
            X_b_top = torch.cat([x_b_raw, torch.full_like(x_b_raw, 4.0), t_b_raw], dim=1)       
            
            # 3. 初始点
            x_i_raw = torch.rand(N_I, 1, device=device) * 4 - 2
            y_i_raw = torch.rand(N_I, 1, device=device) * 8 - 4 
            t_i_raw = torch.zeros(N_I, 1, device=device)
            X_i = torch.cat([x_i_raw, y_i_raw, t_i_raw], dim=1)
            U_i_target = u_init(x_i_raw, y_i_raw, mu).squeeze().detach()

            total_points = N_F + (N_B * 4) + N_I
            
            model = PINN(layers=[3, 10, 10, 10, 10, 10, 1]).to(device)
            optimizer = optim.Adam(model.parameters(), lr=0.001)
            loss_history_gpu = []

            # --- PHASE A: T_train ---
            model.train()
            if torch.cuda.is_available(): torch.cuda.synchronize()
            t_train_start = time.perf_counter()
            
            for epoch in range(EPOCHS):
                optimizer.zero_grad(set_to_none=True) 
                total_loss, loss_components = compute_loss(model, X_f, X_b_left, X_b_right, X_b_bottom, X_b_top, X_i, U_i_target, mu)
                total_loss.backward()
                optimizer.step()
                
                loss_history_gpu.append(total_loss.detach())
                    
            if torch.cuda.is_available(): torch.cuda.synchronize()
            T_train = time.perf_counter() - t_train_start
            
            # 采用和 1D 代码完全一致的 loss stack 方式，确保公平且避免 CPU/GPU 频繁同步
            loss_history_cpu = torch.stack(loss_history_gpu).cpu().numpy().astype(np.float64)
            e_loss = float(loss_history_cpu[-1])
            np.save(f"2d_{METHOD_NAME}_loss_history_mu{mu:.0e}_seed{seed}.npy", loss_history_cpu)

            T_train_per_iter_ms = (T_train * 1e3) / EPOCHS
            T_train_per_iter_point_us = (T_train * 1e6) / (EPOCHS * total_points)
            
            print(f" > Trained: points={total_points}, T_train={T_train:.2f}s, e_loss={e_loss:.3e}")

            # --- PHASE B: T_eval (Separated Warmup + Timing) ---
            model.eval()
            
            with torch.no_grad():
                for _ in range(EVAL_WARMUP):
                    _ = model(net_input_eval)
            
            if torch.cuda.is_available(): torch.cuda.synchronize()
            eval_start = time.perf_counter()
            
            with torch.no_grad():
                for _ in range(EVAL_REPEAT):
                    _ = model(net_input_eval)
                    
            if torch.cuda.is_available(): torch.cuda.synchronize()
            T_eval = (time.perf_counter() - eval_start) / EVAL_REPEAT

            # --- PHASE C: Error Computation ---
            with torch.no_grad():
                u_pred_flat = model(net_input_eval).squeeze().cpu().numpy()
            
            e2, einf = compute_error(true_np, u_pred_flat)
            T_total = T_train + T_eval

            print(f"    -> [mu={mu}] T_eval={T_eval:.6e}s | e2={e2:.3e} | einf={einf:.3e}")

            if SAVE_LHS_PREDICTION:
                df_pred = pd.DataFrame({
                    't': data_mu["t_np"].reshape(-1), 
                    'x': data_mu["x_np"].reshape(-1), 
                    'y': data_mu["y_np"].reshape(-1), 
                    'u': u_pred_flat
                })
                df_pred.to_csv(f"2d_{METHOD_NAME}_U0_predicted_LHS_mu{mu:.0e}_seed{seed}.csv", index=False)
            
            metrics[mu].append({
                'Seed': seed, 'N_test': n_test, 'e_loss': e_loss, 
                'e2': e2, 'einf': einf,
                'T_train': T_train, 'T_eval': T_eval, 'T_total': T_total,
                'T_train_per_iter_ms': T_train_per_iter_ms, 
                'T_train_per_iter_point_us': T_train_per_iter_point_us,
                'total_trained_steps': EPOCHS,
                'total_point_steps': EPOCHS * total_points
            })

    # ================= 4. 输出统计与全面打印 =================
    print("\n" + "="*80)
    print("ALL SEEDS COMPLETED. GENERATING PUBLICATION TABLES...")
    print("="*80 + "\n")

    for mu in MU_LIST:
        df_mu = pd.DataFrame(metrics[mu])
        df_mu.to_csv(f"2d_{METHOD_NAME}_mu{mu:.0e}_Metrics_Summary.csv", index=False)
        
        cols = ['e_loss', 'e2', 'einf', 'T_train', 'T_eval', 'T_total', 'T_train_per_iter_ms', 'T_train_per_iter_point_us']
        stats = {col: mean_std(df_mu[col].values) for col in cols}

        print(f"### Results for 2D PINN Case (mu={mu}) [Mean \\pm Std] ###")
        print(f"e_loss: {stats['e_loss'][0]:.3e} \\pm {stats['e_loss'][1]:.3e}")
        print(f"e_2: {stats['e2'][0]:.3e} \\pm {stats['e2'][1]:.3e}")
        print(f"e_inf: {stats['einf'][0]:.3e} \\pm {stats['einf'][1]:.3e}")
        print(f"T_train (s): {stats['T_train'][0]:.2f} \\pm {stats['T_train'][1]:.2f}")
        print(f"T_eval (s): {stats['T_eval'][0]:.6e} \\pm {stats['T_eval'][1]:.6e}")
        print(f"T_total (s): {stats['T_total'][0]:.2f} \\pm {stats['T_total'][1]:.2f}")
        print(f"T_train/iter (ms): {stats['T_train_per_iter_ms'][0]:.4f} \\pm {stats['T_train_per_iter_ms'][1]:.4f}")
        print(f"T_train/(iter*pt) (us): {stats['T_train_per_iter_point_us'][0]:.4f} \\pm {stats['T_train_per_iter_point_us'][1]:.4f}\n")


Starting 2D PINN Benchmark with Strict LHS-based Timing
Device: cuda | LHS test points: 10000

[mu=0.01] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-02.npy.
[mu=0.001] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-03.npy.
[mu=0.0001] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-04.npy.

Starting 2D PINN for mu=0.01

--- Running Seed: 33 ---
 > Trained: points=16000, T_train=760.50s, e_loss=2.497e+00
    -> [mu=0.01] T_eval=3.027895e-04s | e2=4.989e-01 | einf=6.592e+00

--- Running Seed: 99 ---
 > Trained: points=16000, T_train=787.83s, e_loss=2.386e+00
    -> [mu=0.01] T_eval=3.972554e-04s | e2=3.611e-01 | einf=6.540e+00

--- Running Seed: 202 ---
 > Trained: points=16000, T_train=773.46s, e_loss=1.519e-01
    -> [mu=0.01] T_eval=4.439442e-04s | e2=3.026e-01 | einf=6.487e+00

--- Running Seed: 1234 ---
 > Trained: points=16000, T_train=721.34s, e_loss=1.586e-01
    -> [mu=0.01] T_eval=3.950366e-04s | e2=3.479e-01 | einf=6.486e+00

--- Running Seed: 